# Model Tuning — Grid Search Comparison
เปรียบเทียบ model หลายตัวด้วย Grid Search หาตัวที่ F1 ดีที่สุด

## 1. Imports

In [19]:
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import contractions

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, f1_score,
    roc_auc_score, precision_score, recall_score
)


print('All imports OK')


All imports OK


## 1.5 Constants


In [ ]:
DATA_PATH = '../data/train.csv'
LABEL_COLUMNS_TO_DROP = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

SEARCH_RANDOM_STATE = 42
SEARCH_TEST_SIZE = 0.20
SEARCH_CV_SPLITS = 3
SEARCH_CLASS_WEIGHTS = ['balanced', {0:1, 1:3}]
search_cv = StratifiedKFold(
    n_splits=SEARCH_CV_SPLITS,
    shuffle=True,
    random_state=SEARCH_RANDOM_STATE,
)

WORD_TFIDF_CONFIG = {
    'ngram_range': (1, 2),
    'max_features': 10000,
    'min_df': 2,
    'max_df': 0.9,
    'sublinear_tf': True,
}
CHAR_TFIDF_CONFIG = {
    'analyzer': 'char_wb',
    'ngram_range': (3, 5),
    'max_features': 10000,
    'min_df': 2,
    'sublinear_tf': True,
}
SCALER_CONFIG = {'with_mean': False}

SGD_GRID_PARAM_GRID = [
    {
        'alpha': [1e-5, 1e-4, 1e-3],
        'penalty': ['l2', 'l1'],
        'class_weight': SEARCH_CLASS_WEIGHTS,
        'max_iter': [1000],
        'tol': [1e-4],
    },
    {
        'alpha': [1e-5, 1e-4],
        'penalty': ['elasticnet'],
        'class_weight': SEARCH_CLASS_WEIGHTS,
        'max_iter': [1000],
        'tol': [1e-4],
        'l1_ratio': [0.15, 0.50],
    },
]
SGD_GRID_MODEL_CONFIG = {
    'loss': 'modified_huber',
    'random_state': SEARCH_RANDOM_STATE,
    'n_jobs': None,
}
LR_GRID_PARAM_GRID = [
    {
        'C': [0.03, 0.30, 3.0],
        'class_weight': SEARCH_CLASS_WEIGHTS,
        'solver': ['liblinear'],
        'penalty': ['l2', 'l1'],
        'tol': [1e-4],
    },
    {
        'C': [0.10, 1.0],
        'class_weight': SEARCH_CLASS_WEIGHTS,
        'solver': ['saga'],
        'penalty': ['l2', 'l1'],
        'tol': [1e-4],
    },
]
LR_GRID_MODEL_CONFIG = {
    'max_iter': 1000,
    'random_state': SEARCH_RANDOM_STATE,
    'n_jobs': None,
}

GRID_SEARCH_WORD_VECTORIZER_PATH = 'grid_search_word_vectorizer.pkl'
GRID_SEARCH_CHAR_VECTORIZER_PATH = 'grid_search_char_vectorizer.pkl'
GRID_SEARCH_SCALER_PATH = 'grid_search_scaler.pkl'
GRID_SEARCH_BEST_MODEL_PATH = 'grid_search_best_model.pkl'


## 2. Load & Clean Data

In [21]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=LABEL_COLUMNS_TO_DROP)

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    text = re.sub(r'[^a-z]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['raw_text']   = df['comment_text'].fillna('').astype(str)
df['clean_text'] = df['raw_text'].apply(clean_text)
df = df[df['clean_text'] != ''].copy()
df = df.drop_duplicates(subset=['clean_text']).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(df['toxic'].value_counts())


Dataset shape: (158194, 5)
toxic
0    143038
1     15156
Name: count, dtype: int64


## 3. Feature Engineering

In [22]:
PROFANITY_TERMS = [
    'fuck', 'fucking', 'shit', 'bitch', 'bastard', 'asshole', 'idiot', 'moron',
    'dumb', 'stupid', 'suck', 'crap', 'damn', 'jerk', 'loser', 'trash'
]
IDENTITY_TERMS = [
    'black', 'white', 'gay', 'lesbian', 'transgender', 'trans', 'muslim',
    'jewish', 'christian', 'hispanic', 'asian', 'woman', 'women', 'man', 'men'
]
SECOND_PERSON_TERMS = ['you', 'your', 'yours', 'yourself', 'u']
NEGATION_TERMS      = ['not', 'never', 'no', 'none', 'cannot', 'cant', 'do not']
NON_TOXIC_NEGATION_PATTERNS = [
    r'\bnot\s+(?:stupid|dumb|idiot|moron|trash|wrong|bad|terrible|awful|useless)\b',
    r'\bnot\s+(?:an|a)\s+(?:idiot|moron|loser|bastard|fool)\b',
    r'\bdo\s+not\s+(?:like|love|agree|hate|dislike|attack|insult|blame)\b',
    r'\bcannot\s+(?:hate|blame)\b',
    r'\bnot\s+trying\s+to\s+(?:attack|insult|offend)\b',
]
COMMON_SHORT_TOKENS = {
    'i','me','my','you','your','yours','yourself','it','this','that',
    'a','an','the','is','am','are','was','were','be','to','of','and'
}

def make_term_pattern(terms):
    escaped = sorted((re.escape(t) for t in terms), key=len, reverse=True)
    return re.compile(r'\b(?:' + '|'.join(escaped) + r')\b')

PROFANITY_PATTERN          = make_term_pattern(PROFANITY_TERMS)
IDENTITY_PATTERN           = make_term_pattern(IDENTITY_TERMS)
SECOND_PERSON_PATTERN      = make_term_pattern(SECOND_PERSON_TERMS)
NEGATION_PATTERN           = make_term_pattern(NEGATION_TERMS)
NON_TOXIC_NEGATION_PATTERN = re.compile('|'.join(NON_TOXIC_NEGATION_PATTERNS))

def count_pattern(text, pattern):
    return len(pattern.findall(str(text).lower()))

def repeated_characters_score(text):
    return len(re.findall(r'(.)\1{2,}', str(text).lower()))

def repeated_punctuation_count(text):
    return len(re.findall(r'([!?.,])\1+', str(text)))

def uppercase_ratio(text):
    letters = re.findall(r'[A-Za-z]', str(text))
    if not letters:
        return 0.0
    return sum(1 for c in letters if c.isupper()) / len(letters)

def short_unclear_without_toxic_signal(clean_text, profanity_count):
    tokens = str(clean_text).split()
    content_tokens = [t for t in tokens if t not in COMMON_SHORT_TOKENS]
    too_short = len(tokens) < 3 or len(content_tokens) < 1
    return int(too_short and profanity_count == 0)

# Oangsa features
df['Character Count']                    = df['raw_text'].apply(len)
df['Word Count']                         = df['clean_text'].apply(lambda x: len(x.split()))
df['Exclamation Count']                  = df['raw_text'].str.count('!')
df['Profanity Count']                    = df['clean_text'].apply(lambda x: count_pattern(x, PROFANITY_PATTERN))
df['Strong Toxic Signal Flag']           = (df['Profanity Count'] > 0).astype(int)
df['Second-person Pronoun Count']        = df['clean_text'].apply(lambda x: count_pattern(x, SECOND_PERSON_PATTERN))
df['Repeated Character Pattern Count']  = df['raw_text'].apply(repeated_characters_score)
df['Average Word Length']                = df['clean_text'].apply(
    lambda x: sum(len(w) for w in x.split()) / len(x.split()) if x.split() else 0
)

# Ploy features
df['Uppercase Ratio']                          = df['raw_text'].apply(uppercase_ratio)
df['Question Mark Count']                      = df['raw_text'].str.count(r'\?')
df['Repeated Punctuation Count']               = df['raw_text'].apply(repeated_punctuation_count)
df['Identity-group Term Count']                = df['clean_text'].apply(lambda x: count_pattern(x, IDENTITY_PATTERN))
df['URL Count']                                = df['raw_text'].str.count(r'http\S+|www\S+')
df['Negation Count']                           = df['clean_text'].apply(lambda x: count_pattern(x, NEGATION_PATTERN))
df['Non-toxic Negation Pattern Count']         = df['clean_text'].apply(lambda x: count_pattern(x, NON_TOXIC_NEGATION_PATTERN))
df['Short/Unclear Without Toxic Signal Flag']  = df.apply(
    lambda row: short_unclear_without_toxic_signal(row['clean_text'], row['Profanity Count']), axis=1
)

print('Feature engineering done')

Feature engineering done


## 4. Vectorize + Scale + Split

In [23]:
ENG_FEATURE_COLS = [
    'Character Count', 'Word Count', 'Exclamation Count',
    'Profanity Count', 'Strong Toxic Signal Flag',
    'Second-person Pronoun Count', 'Repeated Character Pattern Count',
    'Average Word Length',
    'Uppercase Ratio', 'Question Mark Count', 'Repeated Punctuation Count',
    'Identity-group Term Count', 'URL Count', 'Negation Count',
    'Non-toxic Negation Pattern Count', 'Short/Unclear Without Toxic Signal Flag'
]

X_raw   = df['raw_text']
X_clean = df['clean_text']
X_eng   = df[ENG_FEATURE_COLS]
y       = df['toxic']

(
    X_raw_train,   X_raw_test,
    X_clean_train, X_clean_test,
    X_eng_train,   X_eng_test,
    y_train,       y_test
) = train_test_split(
    X_raw, X_clean, X_eng, y,
    test_size=SEARCH_TEST_SIZE,
    random_state=SEARCH_RANDOM_STATE,
    stratify=y,
)

word_vec = TfidfVectorizer(**WORD_TFIDF_CONFIG)
char_vec = TfidfVectorizer(**CHAR_TFIDF_CONFIG)

X_word_train = word_vec.fit_transform(X_clean_train)
X_word_test  = word_vec.transform(X_clean_test)
X_char_train = char_vec.fit_transform(X_raw_train)
X_char_test  = char_vec.transform(X_raw_test)

scaler = StandardScaler(**SCALER_CONFIG)
X_eng_train_scaled = scaler.fit_transform(X_eng_train.values)
X_eng_test_scaled  = scaler.transform(X_eng_test.values)

X_train = hstack([X_word_train, X_char_train, csr_matrix(X_eng_train_scaled)], format='csr')
X_test  = hstack([X_word_test,  X_char_test,  csr_matrix(X_eng_test_scaled)], format='csr')

print(f'Train: {len(y_train):,}  |  Test: {len(y_test):,}')
print(f'Feature matrix (train): {X_train.shape}')
print(f'Feature matrix (test) : {X_test.shape}')


Train: 126,555  |  Test: 31,639
Feature matrix (train): (126555, 20016)
Feature matrix (test) : (31639, 20016)


## 5. Grid Search — SGD Classifier

This notebook now uses the same shared environment as the Random Search and Optuna notebooks:
- 3-fold stratified CV
- 20 search candidates per model
- 10k word TF-IDF + 10k char TF-IDF + engineered features
- same class-weight candidate list
- same `loss='modified_huber'` SGD family


In [24]:
grid_sgd = GridSearchCV(
    SGDClassifier(**SGD_GRID_MODEL_CONFIG),
    SGD_GRID_PARAM_GRID,
    scoring='f1',
    cv=search_cv,
    n_jobs=-1,
    verbose=1,
)
grid_sgd.fit(X_train, y_train)

best_sgd = grid_sgd.best_estimator_
y_pred_sgd = best_sgd.predict(X_test)
y_prob_sgd = best_sgd.predict_proba(X_test)[:, 1]

print(f'Best params : {grid_sgd.best_params_}')
print(f'Best CV F1  : {grid_sgd.best_score_:.4f}')
print('\n=== SGD Report ===')
print(classification_report(y_test, y_pred_sgd, target_names=['Not Toxic', 'Toxic']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob_sgd):.4f}')
print(f'F1      : {f1_score(y_test, y_pred_sgd):.4f}')


Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params : {'alpha': 0.0001, 'class_weight': {0: 1, 1: 3}, 'l1_ratio': 0.5, 'max_iter': 1000, 'penalty': 'elasticnet', 'tol': 0.0001}
Best CV F1  : 0.7746

=== SGD Report ===
              precision    recall  f1-score   support

   Not Toxic       0.98      0.97      0.97     28608
       Toxic       0.73      0.81      0.77      3031

    accuracy                           0.95     31639
   macro avg       0.86      0.89      0.87     31639
weighted avg       0.96      0.95      0.95     31639

ROC-AUC : 0.9596
F1      : 0.7684


## 6. Grid Search — Logistic Regression

The LR search now compares `solver in ['liblinear', 'saga']` and `penalty in ['l1', 'l2']` with `max_iter=1000`, matching the other tuning notebooks.


In [25]:
grid_lr = GridSearchCV(
    LogisticRegression(**LR_GRID_MODEL_CONFIG),
    LR_GRID_PARAM_GRID,
    scoring='f1',
    cv=search_cv,
    n_jobs=-1,
    verbose=1,
)
grid_lr.fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)
y_prob_lr = best_lr.predict_proba(X_test)[:, 1]

print(f'Best params : {grid_lr.best_params_}')
print(f'Best CV F1  : {grid_lr.best_score_:.4f}')
print('\n=== Logistic Regression Report ===')
print(classification_report(y_test, y_pred_lr, target_names=['Not Toxic', 'Toxic']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob_lr):.4f}')
print(f'F1      : {f1_score(y_test, y_pred_lr):.4f}')


Fitting 3 folds for each of 40 candidates, totalling 120 fits


KeyboardInterrupt: 

## 7. Compare all result

In [ ]:
results = {
    'Tuned SGD': {
        'F1': f1_score(y_test, y_pred_sgd),
        'Precision': precision_score(y_test, y_pred_sgd),
        'Recall': recall_score(y_test, y_pred_sgd),
        'ROC-AUC': roc_auc_score(y_test, y_prob_sgd),
        'Best CV F1': grid_sgd.best_score_,
    },
    'Tuned LR': {
        'F1': f1_score(y_test, y_pred_lr),
        'Precision': precision_score(y_test, y_pred_lr),
        'Recall': recall_score(y_test, y_pred_lr),
        'ROC-AUC': roc_auc_score(y_test, y_prob_lr),
        'Best CV F1': grid_lr.best_score_,
    },
}

result_df = pd.DataFrame(results).T.round(4)
print('=== Model Comparison ===')
print(result_df.to_string())

result_df['F1'].plot(kind='bar', color=['steelblue', 'tomato'], figsize=(8, 4))
plt.title('Grid Search F1 Score Comparison')
plt.ylabel('F1 Score')
plt.xticks(rotation=15)
plt.ylim(0.5, 0.85)
plt.tight_layout()
plt.show()


## 8. Save Best Model

In [ ]:
# Choose the best tuned model and save method-specific artifacts.
best_f1_sgd = f1_score(y_test, y_pred_sgd)
best_f1_lr  = f1_score(y_test, y_pred_lr)

if best_f1_lr >= best_f1_sgd:
    final_model = best_lr
    print(f'Winner: Logistic Regression (F1={best_f1_lr:.4f})')
else:
    final_model = best_sgd
    print(f'Winner: SGD Classifier (F1={best_f1_sgd:.4f})')

with open(GRID_SEARCH_WORD_VECTORIZER_PATH, 'wb') as f:
    pickle.dump(word_vec, f)
with open(GRID_SEARCH_CHAR_VECTORIZER_PATH, 'wb') as f:
    pickle.dump(char_vec, f)
with open(GRID_SEARCH_SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)
with open(GRID_SEARCH_BEST_MODEL_PATH, 'wb') as f:
    pickle.dump(final_model, f)

print(f'Saved: {GRID_SEARCH_BEST_MODEL_PATH}')
print(f'Saved: {GRID_SEARCH_WORD_VECTORIZER_PATH} / {GRID_SEARCH_CHAR_VECTORIZER_PATH} / {GRID_SEARCH_SCALER_PATH}')
